# Mapping the Electronic Structure Problem for SQD

The foundational problem in quantum chemistry is determining the energy of a given electronic structure.

At its core, this involves solving a complex eigenvalue problem known as the Schrödinger equation:

$$
\hat{\mathcal{H}} \ket{\psi} = E \ket{\psi}
$$

## The Hamiltonian

Here, $\hat{\mathcal{H}}$ represents the **Hamiltonian operator**. In classical mechanics, the Hamiltonian corresponds to the total energy of a system, typically expressed as $\mathcal{H} = T + V$ (kinetic plus potential energy). While in classical contexts this applies to macroscopic objects, in quantum mechanics we deal with microscopic particles like molecules, atoms, and electrons, for which observable quantities (e.g., position, momentum, energy) are no longer definite. Instead, these observables become **operators**, and the states are described by **wave functions** (or **kets**), which encode probabilities.

In quantum chemistry, the Hamiltonian still describes the total energy, but we must account for more complex interactions, including both **one-electron** and **two-electron** terms:

$$
\hat{H} = \sum_{ \substack{p r\\\sigma} } h_{pr} \, \hat{a}^\dagger_{p\sigma} \hat{a}_{r\sigma}
+ \frac{1}{2}
\sum_{ \substack{p r q s\\\sigma \tau} }
h_{prqs} \, 
\hat{a}^\dagger_{p\sigma}
\hat{a}^\dagger_{q\tau}
\hat{a}_{s\tau}
\hat{a}_{r\sigma}
$$

This is the **second-quantized form** of the quantum chemistry Hamiltonian. The terms $h_{pr}$ and $h_{prqs}$ are known as **one-electron** and **two-electron integrals**, respectively. They encode how electrons interact with the nuclei and with each other. The two-electron integrals are particularly difficult to compute, which is why many approximations are often employed.

The operators $\hat{a}$ and $\hat{a}^\dagger$ are **annihilation** and **creation operators** that describe the removal and addition of electrons in specific spin orbitals. For now, you can set these aside—refer to *Helgaker et al.* for a deep dive.

---

## The Wave Function

The symbol $\ket{\psi}$ denotes a **ket**, a mathematical object analogous to a vector. Kets reside in a Hilbert space and are used to calculate probabilities of various quantum states.

### Key points to understand:

- A ket may be referred to by several names: *wave function*, *state vector*, *probability amplitude*, etc.
- Kets are **normalized**, meaning they must have unit norm:

  $$
  \langle \psi | \psi \rangle = \int \psi^*(x) \psi(x) \, dx = 1
  $$

  This is known as the **inner product**, similar to a dot product but involving the complex conjugate transpose (Hermitian adjoint) of the bra vector on the left.

- Basis kets, much like unit vectors $\hat{i}$ and $\hat{j}$ in Euclidean space, are typically **orthogonal**. Together, they form a complete basis for the Hilbert space, which is **complex**, **closed**, and **complete**.
- The coefficients in a ket represent **probability amplitudes**. Their squared magnitudes give the probabilities of different configurations.

In quantum chemistry, the relevant Hilbert space is extended to a **Fock space**, which accommodates varying numbers of electrons. The dimensionality of the Fock space grows exponentially: if there are $M$ spin orbitals, then the Fock space has dimension $2^M$. For a fixed number of electrons $N$, only a subset of these basis states (those with $N$ ones) are physical.

### Example:

$$
\ket{\Psi} = \ket{1,1,0,0}
$$

This ket could represent the electronic configuration of an $H_2$ molecule, where the system is fully described by this occupation number basis state. Let's assume we are working in the STO-3G basis (a minimal level of theory). Each entry in the ket corresponds to a **spin orbital**, which combines spatial orbital information (e.g., 1s, 2s) with spin (up $\alpha$ or down $\beta$).

Since $H_2$ in STO-3G has two spatial orbitals, each capable of housing one spin-up and one spin-down electron, we have four spin orbitals in total.

In the occupation number representation, a `1` indicates that the corresponding spin orbital is occupied by an electron; a `0` means it is unoccupied.

More generally, the wave function is often a **superposition** of occupation number states:

$$
\ket{\Psi} = c_0 \ket{1,1,0,0} + c_1 \ket{1,0,1,0} + c_2 \ket{1,0,0,1} + \cdots
$$

The coefficients $c_n$ are complex probability amplitudes and must satisfy the normalization condition:

$$
\sum_n |c_n|^2 = 1
$$

This wave function represents a superposition of possible electron configurations for the system. Since the system only has 2 electrons, any basis state with more than two ones would be physically invalid.


In [512]:
# Import necessary libraries. For qiskit to work, you need to make an account and get an API key.

import pyscf # quantum chemistry library that builds molecules, makes wave functions, and builds hamiltonians
import pyscf.cc
import pyscf.mcscf
import ffsim
import numpy as np
import matplotlib.pyplot as plt

from shutil import copy
import numpy as np
import pandas as pd
import os
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from glob import glob
import psi4 # Another quantum chemistry library that does the same thing
import sys
sys.path.insert(0, os.path.join(os.path.expanduser('~'),'DDLUCJ/check_amplitudes/'))
from helper_CC_ML_spacial import *
import networkx as nx

In [513]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
 
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
 
from qiskit_addon_sqd.counts import counts_to_arrays
from qiskit_addon_sqd.configuration_recovery import recover_configurations
from qiskit_addon_sqd.fermion import solve_fermion
from qiskit_addon_sqd.subsampling import postselect_and_subsample

## PySCF: Preparing the System for SQD

Below, we explain the key variables of interest in preparing a quantum system using PySCF. This includes how they are obtained, their mathematical form (when applicable), and how they are reused throughout the simulation process.

### Building a Molecule in PySCF

The `gto.Mole()` class initializes a molecule object. The `.build()` method finalizes the object using the specified input parameters, which typically include:

- Atomic coordinates
- Basis set (e.g., `'sto-3g'`, `'6-31g'`)
- Spin information
- Symmetry settings (optional)

```python
from pyscf import gto

mol_pyscf = gto.Mole()
mol_pyscf.atom = 'H 0 0 0; H 0 0 0.74'  # Example: H2 molecule
mol_pyscf.basis = 'sto-3g'
mol_pyscf.spin = 0  # Singlet state (number of unpaired electrons)
mol_pyscf.build()
```
### Active Space, Molecular Orbitals (MOs), Atomic Orbitals (AOs), etc.

- **Molecular Orbitals (MOs)** refer to spatial orbitals—delocalized orbitals formed from linear combinations of atomic orbitals (AOs). They are the orbitals in which electrons reside in a molecule.

- An **active space** is a selected subset of MOs and electrons that are treated explicitly using methods such as **CASSCF** (Complete Active Space Self-Consistent Field) or **CASCI** (Complete Active Space Configuration Interaction).

- The goal of using an active space is to make calculations tractable: treating all orbitals and electrons exactly is computationally prohibitive for anything but the smallest systems. By selecting the most chemically relevant orbitals—typically the **valence orbitals**—and the associated electrons, we retain accuracy where it matters most.

- The **Complete Active Space (cas_pyscf)** includes all possible electron configurations (determinants) within the selected active orbitals and electrons. Orbitals that are not part of the active space are typically:

  - **Frozen** (core orbitals that remain doubly occupied and are excluded from correlation)
  - **Inactive** (fully occupied, but not actively involved in configuration mixing)
  - **Virtual** (unoccupied and excluded from correlation)

> In summary:  
> - The active space = a chosen set of MOs + a selected number of electrons  
> - cas_pyscf includes all configurations within that space  
> - Orbitals outside the cas_pyscf are assumed to be chemically inactive and computationally simplified


In [514]:
## Setting Up the Problem Using PySCF

# Specify molecule properties
open_shell = False
spin_sq = 0

# Build HLi molecule
struct_path = os.path.join(os.path.expanduser('~'),'DDLUCJ/check_amplitudes/diatomics/HLi.xyz')
with open(struct_path,'r') as f:
    text=f.read()

mol_pyscf = pyscf.gto.Mole()
mol_pyscf.verbose = 2

mol_pyscf.build(
    atom=struct_path,
    basis="STO-3G",
    symmetry="c1",
    spin=int(text.split('\n')[1].split()[1])-1,
)

# Define active space
n_frozen = 1
active_space_pyscf = range(n_frozen, mol_pyscf.nao_nr())

## Quantum Chemistry Calculations and Levels of Theory

After building a molecule, we can begin calculating wave functions, extracting molecular integrals, and evaluating other quantities of interest.

Solving the Schrödinger equation (see earlier section) for molecular systems beyond hydrogenic atoms is generally **intractable**. To make progress, various **approximations and numerical techniques** must be employed. These methods typically treat some parts of the system exactly, while approximating others.

This section introduces several essential **levels of theory** used in quantum chemistry:

- Hartree-Fock (HF)
- CASCI / CASSCF
- Multiconfigurational methods
- Møller–Plesset perturbation theory (MP2)
- Coupled Cluster theory (CCSD)

---

### Hartree-Fock (HF) Approximation

The goal of the Hartree-Fock approximation is to compute an approximate **ground state** wave function, denoted as:

$$
\ket{\Phi_{HF}}
$$

The **ground state** refers to the lowest-energy configuration of a molecule. HF uses the **Self-Consistent Field (scf_pyscf)** method, where each electron experiences the **average** interaction from all other electrons rather than specific pairwise interactions.

- The scf_pyscf process is **iterative**, involving repeated diagonalization of the **Fock matrix**.
- It scales approximately as $O(n^3)$, where *n* is the system size.
- Variants of HF include:
  - **RHF** (Restricted HF): Constrains the wave function to be an eigenfunction of the spin operator.
  - **UHF** (Unrestricted HF): Allows different spatial orbitals for spin-up and spin-down electrons.

In PySCF, the result of an RHF calculation includes:
- Optimized **spin orbitals**
- Electron **occupation numbers**

These define a **reference state** that serves as the starting point for more advanced methods.

> ⚠️ **Note:** Hartree-Fock does **not** handle **strongly correlated systems** well, as it only captures exchange interactions and neglects electron correlation.

---

### Multiconfigurational Self-Consistent Field (MCSCF)

The goal of **CASCI/CASSCF** and other MCSCF methods is to improve upon HF by incorporating **electron correlation** more accurately. This is done by allowing multiple electron configurations (determinants) to contribute to the wave function, particularly in an **active space** of important orbitals and electrons.

- **CASCI** (Complete Active Space Configuration Interaction): Solves for the exact wave function and energy **within a fixed active space**.
- **CASSCF** (Complete Active Space scf_pyscf): Optimizes both the coefficients of the configurations *and* the orbitals themselves.

In PySCF, the `cas_pyscf` object constructed by CASCI or CASSCF contains methods to compute:
- **One-electron integrals** (`h1cas`)
- **Two-electron integrals** (`h2cas`)

These integrals are crucial for downstream quantum algorithms and simulation.

Key components:
- `hcore_pyscf`: The **one-electron Hamiltonian matrix**
- `nuclear_repulsion_energy`: Constant term representing **nuclear–nuclear repulsion**
- `eri_pyscf`: The **electron repulsion integral**, representing two-electron interactions

These elements allow the construction of the effective Hamiltonian used in multiconfigurational and post-HF methods.


In [515]:
# Build reference wave function using RHF
scf_pyscf = pyscf.scf.RHF(mol_pyscf).run()
num_orbitals_pyscf = len(active_space_pyscf)

# Identify number of electrons in active sapce using the occupation numbers from the RHF
n_electrons_pyscf = int(sum(scf_pyscf.mo_occ[active_space_pyscf]))
num_elec_a_pyscf = (n_electrons_pyscf + mol_pyscf.spin) // 2
num_elec_b_pyscf = (n_electrons_pyscf - mol_pyscf.spin) // 2

# Use Complete Active Space Configuration Interaction method to get the
# one and two electron integrals
cas_pyscf = pyscf.mcscf.CASCI(scf_pyscf, num_orbitals_pyscf, (num_elec_a_pyscf, num_elec_b_pyscf))
mo_pyscf = cas_pyscf.sort_mo(active_space_pyscf, base=0)
hcore_pyscf, nuclear_repulsion_energy = cas_pyscf.get_h1cas(mo_pyscf)
eri_pyscf = pyscf.ao2mo.restore(1, cas_pyscf.get_h2cas(mo_pyscf), num_orbitals_pyscf)

# Compute exact energy
exact_energy_pyscf = cas_pyscf.run().e_tot


## Coupled Cluster Singles and Doubles (CCSD)

One of the most critical components in mapping the quantum chemistry problem to quantum hardware—particularly for algorithms like LUCJ—is the computation of **T1 and T2 excitation amplitudes**. These amplitudes are essential for initializing the system.

### Mathematical Formulation

The CCSD wavefunction is defined as:

$$
\ket{\Psi_{\text{CCSD}}} = e^{\hat{T}_1 + \hat{T}_2} \ket{\Phi_{\text{RHF}}}
$$

- $\ket{\Psi_{\text{CCSD}}}$: The **correlated** CCSD wavefunction
- $\hat{T}_1$, $\hat{T}_2$: Excitation operators built from **T1 and T2 amplitudes**
- $\ket{\Phi_{\text{RHF}}}$: The **reference wavefunction**, obtained from a prior **RHF** calculation

These excitation operators account for:
- $\hat{T}_1$: **Single excitations** (electron moves from one occupied to one virtual orbital)
- $\hat{T}_2$: **Double excitations** (two electrons are simultaneously excited)

---

### What Are T1 and T2 Amplitudes?

| Concept       | Meaning                                                                                                                             |
| ------------- | ----------------------------------------------------------------------------------------------------------------------------------- |
| Amplitude $t$ | A coefficient in second quantization that encodes how much a particular excited configuration contributes to the wavefunction       |
| Physically    | Measures the **deviation from the mean-field Hartree-Fock** result—i.e., how much correction is needed beyond HF                    |
| Numerically   | Obtained by solving a set of **nonlinear equations** derived from projecting the similarity-transformed Schrödinger equation        |

These amplitudes are **not** extracted directly from integrals; rather, they are solutions to the following projected equations:

$$
\bra{\Phi_i^a} e^{-\hat{T}} \hat{H} e^{\hat{T}} \ket{\Phi_{\text{RHF}}} = 0
$$

$$
\bra{\Phi_{ij}^{ab}} e^{-\hat{T}} \hat{H} e^{\hat{T}} \ket{\Phi_{\text{RHF}}} = 0
$$

- $\Phi_i^a$: A single excitation (electron *i* to orbital *a*)
- $\Phi_{ij}^{ab}$: A double excitation (electrons *i, j* to orbitals *a, b*)

These equations are highly nonlinear and typically solved iteratively.

---

### The Role of Integrals and Density Matrix

While **T1 and T2 amplitudes** are not computed directly from integrals, the Hamiltonian $\hat{H}$ used in the CCSD equations *does* contain the **one-electron** and **two-electron integrals**, making them indispensable.

Additionally:

- **J (Coulomb operator)** and **K (Exchange operator)** are constructed during the process.
- Both **J** and **K** depend on the **Density Matrix** $D$, which itself is constructed from the T1 and T2 amplitudes.

Thus, the flow of information is:

1. Start from one- and two-electron integrals
2. Solve for T1 and T2 amplitudes using similarity-transformed Hamiltonian
3. Use T1 and T2 to compute the density matrix
4. From the density matrix, construct J and K operators
5. Build the correlated energy and wavefunction

> ✅ **Summary:** T1 and T2 amplitudes are the backbone of CCSD—they measure correlation beyond HF and enable quantum-classical mappings like LUCJ.


In [516]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd_pyscf = pyscf.cc.CCSD(
    scf_pyscf, frozen=[i for i in range(mol_pyscf.nao_nr()) if i not in active_space_pyscf]
).run()
t1_pyscf = ccsd_pyscf.t1
t2_pyscf = ccsd_pyscf.t2

## Local Unitary Cluster Jastrow (UCJ) Ansatz and the Role of J and K Operators

The **Local Unitary Cluster Jastrow (UCJ)** ansatz is a hardware-efficient variational form used in quantum chemistry simulations. It is designed to capture **electron correlation** by applying a structured, local transformation on top of a **Hartree-Fock reference state**. In particular, the UCJ ansatz uses **parameterized two-body entanglers** based on **excitation amplitudes** (T1 and T2) from classical CCSD calculations.

---

### UCJ Ansatz Overview

The UCJ ansatz applies **local unitary gates** to introduce electron-electron correlations via **clustered Jastrow-like terms**. The structure:

- Preserves **spin symmetry** (via `UCJOpSpinBalanced`)
- Is **locally connected**, making it more feasible for near-term quantum devices
- Uses **parameterized pairs of orbitals** to restrict entanglement to physically meaningful regions

Mathematically, the UCJ operator looks like:

The LUCJ ansatz is a specialized form of the general unitary cluster Jastrow (UCJ) ansatz, which has the form

$$
  \lvert \Psi \rangle = \prod_{\mu=1}^L e^{\hat{K}_\mu} e^{i \hat{J}_\mu} e^{-\hat{K}_\mu} | \Phi_0 \rangle
$$

where $\lvert \Phi_0 \rangle$ is a reference state, often taken to be the Hartree-Fock state, and the $\hat{K}_\mu$ and $\hat{J}_\mu$ have the form

$$
\hat{K}_\mu = \sum_{pq, \sigma} K_{pq}^\mu \, \hat{a}^\dagger_{p \sigma} \hat{a}^{\phantom{\dagger}}_{q \sigma}
\;,\;
\hat{J}_\mu = \sum_{pq, \sigma\tau} J_{pq,\sigma\tau}^\mu \, \hat{n}_{p \sigma} \hat{n}_{q \tau}
\;,
$$

where we have defined the number operator

$$
\hat{n}_{p \sigma} = \hat{a}^\dagger_{p \sigma} \hat{a}^{\phantom{\dagger}}_{p \sigma}.
$$

The number operator acts on a ket and returns the occupation number associated to an indexed position.

---

### J and K Operators in Context

The **Coulomb (J)** and **Exchange (K)** operators originate in **Hartree-Fock theory** and classical post-HF methods:

| Operator | Description |
|---------|-------------|
| **J**   | The Coulomb operator captures **classical repulsion** between electrons in different orbitals. Think of it as the mean-field electrostatic interaction. |
| **K**   | The Exchange operator arises due to **fermionic antisymmetry** — it captures the quantum mechanical effect of electron indistinguishability and spin correlation. It’s non-classical and crucial in spin-balanced systems. |

Both **J** and **K** depend on the **one-particle density matrix** $D$, which in turn is constructed from the **T1 and T2 amplitudes**. These operators help shape the effective Hamiltonian used in the UCJ construction.

---


In [517]:
n_reps = 1
# Define interaction pairs for local entanglement
alpha_alpha_indices = [(p, p + 1) for p in range(num_orbitals_pyscf - 1)]
alpha_beta_indices = [(p, p) for p in range(0, num_orbitals_pyscf, 4)]

# Create the UCJ operator using CCSD excitation amplitudes
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2_pyscf,
    t1=t1_pyscf,
    n_reps=n_reps,
    interaction_pairs=(alpha_alpha_indices, alpha_beta_indices),
)
 
nelec = (num_elec_a_pyscf, num_elec_b_pyscf)
 
# create an empty quantum circuit
qubits = QuantumRegister(2 * num_orbitals_pyscf, name="q")
circuit = QuantumCircuit(qubits)
 
# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(num_orbitals_pyscf, nelec), qubits)
 
# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()

# From PySCF to Psi4

In order to implement the **DDLUCJ** Method, we must transition this code to Psi4. We are interested in the following parameters:

- [x] Nuclear Repulsion Energy
- [x] Exact Energy
- [x] Molecular Orbitals (active space)
- [x] Number of Electrons $\alpha$ and $\beta$
- [ ] One Electron Integrals
- [ ] Two Electron Integrals
- [x] T1 Amplitudes
- [x] T2 Amplitudes

Some things to consider: 

* Symmetries, like `dooh` are not implemented in Psi4, so they will not work when buidling a molecule
* The $H_2$ molecule in the STO-3G basis does not work on Psi4 (idk why)
* Be careful with freezing, the dimensions of the integrals and number of electrons may be off
* The one and two electron integrals appear to contain the same information, but the arrays are ordered differently
* use np.isclose() to compare the matrices

In [518]:
# === 1. Molecule Setup ===

xyz_path = os.path.join(os.path.expanduser("~"), "DDLUCJ", "check_amplitudes", "diatomics", "HLi.xyz")

with open(xyz_path, 'r') as f:
    xyz_text = f.read()

qmol = psi4.qcdb.Molecule.from_string(xyz_text, dtype='xyz')
mol = psi4.geometry(qmol.create_psi4_string_from_molecule() + "\nsymmetry c1\n")

psi4.core.clean()
psi4.core.be_quiet()

# === 2. Set Options and Run RHF ===

psi4.set_options({
    'basis': 'STO-3G',
    'scf_type': 'pk',
    'reference': 'rhf',
    'mp2_type': 'conv',
    'e_convergence': 1e-8,
    'd_convergence': 1e-8,
    'print_mos': True
})

rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)

# === 3. Use HelperCCEnergy ===

A = HelperCCEnergy(mol, rhf_e, scf_wfn, freeze_core=True)
A.compute_energy()

# === 4. Define Active Space ===
n_frozen = A.nfzc
nmo = A.nmo
active_orbitals = list(range(n_frozen, nmo))

num_orbitals = len(active_orbitals)
n_elec_total = A.ndocc * 2

active_elec = n_elec_total - 2 * n_frozen
num_elec_a = int((active_elec + mol.multiplicity() - 1) // 2)
num_elec_b = int((active_elec - mol.multiplicity() + 1) // 2)

Computing RHF reference.


/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


Cutting 1 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 6 basis functions.
(6, 6)
(5, 5)
Building initial guess...

..initialized CCSD in 0.053 seconds.

CCSD Iteration   0: CCSD correlation = -0.012592560801913   dE =  1.25926E-02   MP2
CCSD Iteration   1: CCSD correlation = -0.017024114490341   dE = -4.43155E-03   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.018703334087821   dE = -1.67922E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.019809445227367   dE = -1.10611E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.020006127884664   dE = -1.96683E-04   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.020068758226443   dE = -6.26303E-05   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.020057947032657   dE =  1.08112E-05   DIIS = 5
CCSD Iteration   7: CCSD correlation = -0.020060378113906   dE = -2.43108E-06   DIIS = 6
CCSD Iteration   8: CCSD correlation = -0.020059255375596   dE =  1.1

/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:552: RuntimeWarning: divide by zero encountered in log10
  self.t2mag=np.log10(np.absolute(self.t2))
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:593: RuntimeWarning: divide by zero encountered in log10
  self.Jia1mag=np.log10(np.absolute(self.Jia1))
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:594: RuntimeWarning: divide by zero encountered in log10
  self.Jia2mag=np.log10(np.absolute(self.Jia2))
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:595: RuntimeWarning: divide by zero encountered in log10
  self.Kia1mag=np.log10(np.absolute(self.Kia1))
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:596: RuntimeWarning: divide by zero encountered in log10
  self.Kia2mag=np.log10(np.absolute(self.Kia2))


In [519]:
## Comparing PySCF vs Psi4 Variables

print(active_orbitals)
print(num_orbitals)
print(active_elec)
print(num_elec_a)
print(num_elec_b)

[1, 2, 3, 4, 5]
5
2
1
1


In [520]:
print(active_space_pyscf)
print(num_orbitals_pyscf)
print(n_electrons_pyscf)
print(num_elec_a_pyscf)
print(num_elec_b_pyscf)

range(1, 6)
5
2
1
1


In [521]:
print(mol_pyscf.spin)               # PySCF
print(mol.multiplicity())           # Psi4
print(mol_pyscf.nao_nr())           # PySCF
print(A.nmo)                        # Psi4


0
1
6
6


## Calculating frozen core energy to get the `exact_energy` later

In [522]:
# Get MO coefficients
Ca = scf_wfn.Ca()

# Prepare integrals
basis = scf_wfn.basisset()
mints = psi4.core.MintsHelper(basis)

# One-electron AO integrals <i|T + V|j>
Hao = mints.ao_kinetic()
Hao.add(mints.ao_potential())

# Transform to MO basis
Hmo = psi4.core.triplet(Ca, Hao, Ca, True, False, False)

# Number of frozen orbitals
Nf = n_frozen

# ERIs in MO basis
Rm = mints.mo_eri(Ca, Ca, Ca, Ca).np

# Compute frozen core energy
efc = (
    2 * np.trace(Hmo.np[:Nf, :Nf]) +
    2 * np.einsum('iijj', Rm[:Nf, :Nf, :Nf, :Nf]) -
    np.einsum('ijij', Rm[:Nf, :Nf, :Nf, :Nf])
)

nuclear_repulsion_energy = efc + mol.nuclear_repulsion_energy()
print("Frozen core energy:", nuclear_repulsion_energy)


Frozen core energy: -6.8017826860055415


## Getting the integrals

In [523]:
# === 5. Extract Active-Space Integrals ===

hcore = A.H1[np.ix_(active_orbitals, active_orbitals)]
eri = A.MO  # Already active-space sliced

# === 6. Use CCSD Final Energy as "Exact" Reference ===

exact_energy = A.FinalEnergy + rhf_e

# === 7. Psi4 Built-in CCSD for Comparison ===

psi4.set_options({'freeze_core': True})
ccsd_e, ccsd_wfn = psi4.energy('ccsd', return_wfn=True)

# === 8. Extract T1 and T2 Amplitudes ===

t1 = A.t1
t2 = A.t2

In [524]:
# === 5. Extract Active-Space Integrals ===

# Get 1-electron AO integrals
h_ao = mints.ao_kinetic()
h_ao.add(mints.ao_potential())

# Transform to MO basis
# Note: Ca is the full MO coefficient matrix from scf_wfn
h_mo = psi4.core.triplet(Ca, h_ao, Ca, True, False, False)

# Slice to active orbitals
hcore = h_mo.np[np.ix_(active_orbitals, active_orbitals)]

# 2-electron integrals are still taken from HelperCCEnergy (already active-space sliced)
eri = A.MO

# === 6. Use CCSD Final Energy as "Exact" Reference ===

exact_energy = A.FinalEnergy + rhf_e

# === 7. Psi4 Built-in CCSD for Comparison ===

psi4.set_options({'freeze_core': True})
ccsd_e, ccsd_wfn = psi4.energy('ccsd', return_wfn=True)

# === 8. Extract T1 and T2 Amplitudes ===

t1 = A.t1
t2 = A.t2


In [525]:
## Comparing PySCF vs Psi4 Variables
print("H1 shape:                       ", hcore.shape)
print("ERI shape:                      ", eri.shape)
print("Active electrons: (α, β)        =", (num_elec_a, num_elec_b))
print("Exact (CCSD) Energy:            ", exact_energy)


H1 shape:                        (5, 5)
ERI shape:                       (5, 5, 5, 5)
Active electrons: (α, β)        = (1, 1)
Exact (CCSD) Energy:             -7.88225445232408


In [526]:
print("H1 shape:                       ", hcore_pyscf.shape)
print("ERI shape:                      ", eri_pyscf.shape)
print("Active electrons: (α, β)        =", (num_elec_a_pyscf, num_elec_b_pyscf))
print("Exact (CCSD) Energy:            ", exact_energy_pyscf)


H1 shape:                        (5, 5)
ERI shape:                       (5, 5, 5, 5)
Active electrons: (α, β)        = (1, 1)
Exact (CCSD) Energy:             -7.882254452694326


In [558]:
print(np.round(t1-t1_pyscf, 8))

# Seems like only one element differs noticeably for each excitation amplitude

[[-7.697139e-02  0.000000e+00  0.000000e+00  1.000000e-08]]


In [559]:
t2

array([[[[-0.03609396,  0.        ,  0.        , -0.05949089],
         [ 0.        , -0.02775828,  0.        ,  0.        ],
         [ 0.        ,  0.        , -0.02775828,  0.        ],
         [-0.05949089,  0.        ,  0.        , -0.11467003]]]])

In [561]:
np.round(t2_pyscf,8)

array([[[[-0.03609396, -0.        ,  0.        ,  0.05949092],
         [-0.        , -0.02775828,  0.        , -0.        ],
         [ 0.        ,  0.        , -0.02775828, -0.        ],
         [ 0.05949092, -0.        , -0.        , -0.11467003]]]])

## Correcting integral ordering between packages

`hcore` and `eri` are multidimensional arrrays. The data they contain is arranged via the ordering based on their atomic and molecular orbitals.

The one and two electron inegrals are first calculated in the *Atomic Orbital* Basis. We then transform these integrals into the molecualr basis using Molecular coefficients MO, C

$$ \psi_i (\textbf{r}) = \sum_{\mu} C_{\mu i} \chi_{\mu} (\textbf{r})
$$

wher C are the molecular coefficints. They transform the slater determinants from the AO basis to the MO basis.

Since the electron integrals are matrices, we transform their basis in the usual way:

$$ h_{ij}^{MO} = \sum_{\mu \nu} C_{\mu i}^* C_{\nu j } (h_{\mu \nu} + V_{core})$$

or equivalently $${h}^{MO} = C^\dagger h C$$ taking h and C to be matrices


In [528]:
hcore-np.round(hcore_pyscf,8)

array([[-0.72254934, -0.08159119,  0.        ,  0.        ,  0.07164109],
       [-0.08159119, -0.76970484,  0.        ,  0.        , -0.09879701],
       [ 0.        ,  0.        , -0.78281879,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , -0.78281879,  0.        ],
       [ 0.07164109, -0.09879701,  0.        ,  0.        , -0.71508344]])

In [529]:
np.isclose(eri,np.round(eri_pyscf,8))

array([[[[ True, False,  True,  True,  True],
         [False, False,  True,  True, False],
         [ True,  True, False,  True,  True],
         [ True,  True,  True, False,  True],
         [ True, False,  True,  True, False]],

        [[False, False,  True,  True, False],
         [ True, False,  True,  True,  True],
         [ True,  True, False,  True,  True],
         [ True,  True,  True, False,  True],
         [False, False,  True,  True, False]],

        [[ True,  True, False,  True,  True],
         [ True,  True, False,  True,  True],
         [ True, False,  True,  True,  True],
         [ True,  True,  True,  True,  True],
         [ True,  True, False,  True,  True]],

        [[ True,  True,  True, False,  True],
         [ True,  True,  True, False,  True],
         [ True,  True,  True,  True,  True],
         [ True, False,  True,  True,  True],
         [ True,  True,  True, False,  True]],

        [[ True, False,  True,  True, False],
         [False, False,  T

## MO Coefficients

In [530]:
print("PySCF AO labels:")
for i, l in enumerate(mol_pyscf.ao_labels()):
    print(i, l)

print("Psi4 AO centers:")
for i in range(mints.basisset().nbf()):
    center = mints.basisset().function_to_center(i)
    print(i, mol.symbol(center))


PySCF AO labels:
0 0 H 1s    
1 1 Li 1s    
2 1 Li 2s    
3 1 Li 2px   
4 1 Li 2py   
5 1 Li 2pz   
Psi4 AO centers:
0 H
1 LI
2 LI
3 LI
4 LI
5 LI


In [531]:
np.isclose(np.array(scf_wfn.epsilon_a()), scf_pyscf.mo_energy, atol=1e-8)

array([ True,  True,  True,  True,  True,  True])

In [532]:
C_pyscf = np.round(scf_pyscf.mo_coeff,5)
C_pyscf

array([[ 0.0046 ,  0.54846, -0.13939,  0.     ,  0.     ,  1.19029],
       [ 0.99124, -0.16779, -0.20991,  0.     ,  0.     ,  0.09214],
       [ 0.03265,  0.4543 ,  0.7997 , -0.     , -0.     , -0.70697],
       [-0.     ,  0.     , -0.     ,  1.     , -0.     ,  0.     ],
       [-0.     ,  0.     ,  0.     ,  0.     ,  1.     , -0.     ],
       [ 0.00639, -0.3464 ,  0.61221,  0.     , -0.     ,  0.9827 ]])

In [533]:
C_Psi4 = np.round(scf_wfn.Ca().to_array().copy(),5)
C_Psi4

array([[ 0.0046 ,  0.54846,  0.13939,  0.     ,  0.     ,  1.19029],
       [ 0.99124, -0.16779,  0.20991,  0.     ,  0.     ,  0.09214],
       [ 0.03265,  0.4543 , -0.7997 ,  0.     ,  0.     , -0.70697],
       [ 0.00639, -0.3464 , -0.61221,  0.     ,  0.     ,  0.9827 ],
       [-0.     , -0.     ,  0.     ,  1.     ,  0.     , -0.     ],
       [-0.     , -0.     ,  0.     ,  0.     ,  1.     , -0.     ]])

In [534]:
np.isclose(np.round(scf_pyscf.mo_coeff,8),np.round(scf_wfn.Ca().to_array().copy(),8))

array([[ True,  True, False,  True,  True,  True],
       [ True,  True, False,  True,  True,  True],
       [ True,  True, False,  True,  True,  True],
       [False, False, False, False,  True, False],
       [ True,  True,  True, False, False,  True],
       [False, False, False,  True, False, False]])

In [554]:
P = [0,1,2,4,5,3]  # PySCF AO → Psi4 AO
C_Psi4_prime = C_Psi4[P,:]
C_Psi4_prime

array([[ 0.0046 ,  0.54846,  0.13939,  0.     ,  0.     ,  1.19029],
       [ 0.99124, -0.16779,  0.20991,  0.     ,  0.     ,  0.09214],
       [ 0.03265,  0.4543 , -0.7997 ,  0.     ,  0.     , -0.70697],
       [-0.     , -0.     ,  0.     ,  1.     ,  0.     , -0.     ],
       [-0.     , -0.     ,  0.     ,  0.     ,  1.     , -0.     ],
       [ 0.00639, -0.3464 , -0.61221,  0.     ,  0.     ,  0.9827 ]])

In [555]:
np.isclose(C_Psi4_prime, C_pyscf)

array([[ True,  True, False,  True,  True,  True],
       [ True,  True, False,  True,  True,  True],
       [ True,  True, False,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [ True,  True, False,  True,  True,  True]])

In [556]:
# Match signs column by column
signs = np.sign(np.sum(C_pyscf * C_Psi4_prime, axis=0))
C_Psi4_prime = C_Psi4_prime * signs  # broadcast sign to each column

np.allclose(C_Psi4_prime, C_pyscf)

True

## Reording the Hao from Psi4 in the AO basis.

In [538]:
P = [0,1,2,4,5,3]
Hao_prime = Hao.np[:,[0,1,2,4,5,3]]
Hao_prime = Hao_prime[[0,1,2,4,5,3],:]
Hao_prime

array([[-1.46272923, -0.30932533, -0.69986273,  0.        ,  0.        ,
         0.84366962],
       [-0.30932533, -4.73905626, -1.06486519,  0.        ,  0.        ,
         0.01631548],
       [-0.69986273, -1.06486519, -1.39703981,  0.        ,  0.        ,
         0.12275254],
       [ 0.        ,  0.        ,  0.        , -1.13680162,  0.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        , -1.13680162,
         0.        ],
       [ 0.84366962,  0.01631548,  0.12275254,  0.        ,  0.        ,
        -1.23545179]])

In [539]:
Hao_pyscf = mol_pyscf.intor('int1e_kin') + mol_pyscf.intor('int1e_nuc')

np.isclose(Hao_prime, Hao_pyscf)

array([[ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True]])

In [540]:
# Convert NumPy array to Psi4 Matrix
C_prime = psi4.core.Matrix.from_array(C_Psi4_prime[:,1:])
Hao_prime = psi4.core.Matrix.from_array(Hao_prime)
# Transform the AO Hamiltonian into the MO basis
Hmo_prime = psi4.core.triplet(C_prime, Hao_prime, C_prime,True, False, False)
# Get ERIs in MO basis using your corrected MO coeffs
Rm_prime = mints.mo_eri(C_prime, C_prime, C_prime, C_prime)


In [541]:
np.isclose(eri_pyscf,Rm_prime.np, atol=1e-6)

array([[[[False, False,  True,  True, False],
         [False, False,  True,  True, False],
         [ True,  True, False,  True,  True],
         [ True,  True,  True, False,  True],
         [False, False,  True,  True, False]],

        [[False, False,  True,  True, False],
         [False, False,  True,  True, False],
         [ True,  True, False,  True,  True],
         [ True,  True,  True, False,  True],
         [False, False,  True,  True, False]],

        [[ True,  True, False,  True,  True],
         [ True,  True, False,  True,  True],
         [False, False,  True,  True, False],
         [ True,  True,  True,  True,  True],
         [ True,  True, False,  True,  True]],

        [[ True,  True,  True, False,  True],
         [ True,  True,  True, False,  True],
         [ True,  True,  True,  True,  True],
         [False, False,  True,  True, False],
         [ True,  True,  True, False,  True]],

        [[False, False,  True,  True, False],
         [False, False,  T

In [542]:
Hmo_prime.np

array([[-1.49679894,  0.03319519,  0.        ,  0.        , -0.05570819],
       [ 0.03319519, -1.12627223,  0.        ,  0.        ,  0.03064741],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [-0.05570819,  0.03064741,  0.        ,  0.        , -0.9491192 ]])

In [543]:
np.round(hcore_pyscf, 8)

array([[-0.77423279,  0.04839889, -0.        ,  0.        , -0.12734728],
       [ 0.04839889, -0.35656556, -0.        , -0.        ,  0.06814968],
       [-0.        , -0.        , -0.35398283,  0.        , -0.        ],
       [ 0.        , -0.        ,  0.        , -0.35398283, -0.        ],
       [-0.12734728,  0.06814968, -0.        , -0.        , -0.23403567]])

## `hcore` depends on V-core which depends on the exchange and coulomb matrices

This does not work (yet)!!!

In [544]:
ncore = 1
C_ao = C_Psi4_prime        # AO → MO coefficients
C_core = C_ao[:, :ncore]            # shape (nAO, ncore)
C_core_p4 = psi4.core.Matrix.from_array(C_core)

# --- build J & K from those core orbitals ------------
jk = psi4.core.JK.build(scf_wfn.basisset())    # un-initialised libJK
jk.initialize()

jk.C_left_add(C_core_p4)            # left == right → ρ = C Cᵀ
jk.C_right_add(C_core_p4)
jk.compute()                        # do the integrals
jk.C_clear()                        # tidy if you’ll reuse jk

J_core = jk.J()[0]                  # first (and only) block
K_core = jk.K()[0]

V_core_psi4 = 2.0 * J_core.np - K_core.np    # RHF veff

In [545]:
Hao_prime.add(psi4.core.Matrix.from_array(V_core_psi4))
Hmo_prime = psi4.core.triplet(C_prime, Hao_prime, C_prime,True, False, False)

In [546]:
Hmo_prime.np

array([[-0.91250838,  0.18838591,  0.        ,  0.        , -0.08086291],
       [ 0.18838591, -0.41871411,  0.        ,  0.        ,  0.28393605],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [-0.08086291,  0.28393605,  0.        ,  0.        ,  0.61890406]])

In [547]:
np.round(hcore_pyscf, 8)

array([[-0.77423279,  0.04839889, -0.        ,  0.        , -0.12734728],
       [ 0.04839889, -0.35656556, -0.        , -0.        ,  0.06814968],
       [-0.        , -0.        , -0.35398283,  0.        , -0.        ],
       [ 0.        , -0.        ,  0.        , -0.35398283, -0.        ],
       [-0.12734728,  0.06814968, -0.        , -0.        , -0.23403567]])

In [548]:
np.allclose(Hmo_prime.np, np.round(hcore_pyscf, 8), atol=1e-3)

False

In [549]:
ncore = 1
mo_core = C_pyscf[:, :ncore]
# === Step 2: Build core density matrix ===
D_core = 2.0 * mo_core @ mo_core.T

# === Step 3: Get core potential ===
V_core = scf_pyscf.get_veff(mol, D_core)

In [550]:
np.isclose(V_core_psi4,V_core, atol=1e-3)

array([[ True, False,  True, False,  True, False],
       [False,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [False,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True],
       [False,  True,  True,  True,  True,  True]])

In [551]:
V_core

array([[ 6.57807925e-01,  6.77284056e-02,  2.88533657e-01,
         0.00000000e+00,  0.00000000e+00, -3.64497043e-01],
       [ 6.77284056e-02,  1.66983512e+00,  2.52377907e-01,
         0.00000000e+00,  0.00000000e+00, -1.53549561e-03],
       [ 2.88533657e-01,  2.52377907e-01,  7.39027554e-01,
         0.00000000e+00,  0.00000000e+00, -1.72634694e-04],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         7.82820870e-01,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  7.82820870e-01,  0.00000000e+00],
       [-3.64497043e-01, -1.53549561e-03, -1.72634694e-04,
         0.00000000e+00,  0.00000000e+00,  7.82778230e-01]])

In [552]:
V_core_psi4

array([[ 6.58072113e-01,  6.65913753e-02,  2.88382118e-01,
        -3.64514714e-01,  0.00000000e+00, -1.26474280e-04],
       [ 6.65913753e-02,  1.66985658e+00,  2.52382782e-01,
         8.16892900e-04,  0.00000000e+00, -2.35289589e-03],
       [ 2.88382118e-01,  2.52382782e-01,  7.39042900e-01,
         1.73291304e-04,  0.00000000e+00, -3.48779011e-04],
       [-3.64514714e-01,  8.16892900e-04,  1.73291304e-04,
         7.82797379e-01,  0.00000000e+00,  3.57034098e-06],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  7.82836733e-01,  0.00000000e+00],
       [-1.26474280e-04, -2.35289589e-03, -3.48779011e-04,
         3.57034098e-06,  0.00000000e+00,  7.82827399e-01]])

In [553]:
# Check max absolute difference
np.max(np.abs(V_core - V_core_psi4))


np.float64(0.36451471418676207)